<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week6/Day3/Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT

### Deliverables
- **Padding Choice**: I chose `max_length=24`. This value is optimal because it slightly exceeds the length of my example sentence (around 10 tokens), avoiding truncation while minimizing unnecessary memory consumption compared to a length of 512.
- **Highlighted Tokens**: [CLS] (ID 101) marks the start, [SEP] (ID 102) marks the end, and [PAD] (ID 0) tokens complete the vector up to 24.

In [1]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Exercise 1: Replacing TODO with a sample sentence
sample_sentence = "Hugging Face is creating amazing tools for NLP!"
print(f"Original sentence: {sample_sentence}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original sentence: Hugging Face is creating amazing tools for NLP!


In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


### Exercise 1 reflection
- **How [CLS] and [SEP] behave**: The `[CLS]` (Classification) token is added at the beginning of the sequence; its final hidden state is often used as an aggregate representation of the sentence for classification tasks. The `[SEP]` (Separator) token serves to mark the end of a sentence or to separate two distinct sentences in an input pair.
- **Attention mask**: The attention mask contains 1s for real tokens and 0s for padding tokens (`[PAD]`). During the self-attention calculation, the model ignores positions marked as 0, preventing calculations from being influenced by empty padding.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Exercise 2: Testing a specific sentence
sentence = "I absolutely love how easy it is to use pre-trained models!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence: {sentence}")
print(f"Result: {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: I absolutely love how easy it is to use pre-trained models!
Result: [{'label': 'POSITIVE', 'score': 0.9996839761734009}]


### Exercise 2 reflection
- **Expectation**: Yes, the label matches my expectations (POSITIVE) because the sentence contains positive terms like 'love' and 'easy'.
- **Confidence**: The score (usually > 0.99) indicates very high confidence. This means the model is almost certain that the sentiment is positive based on the data it was fine-tuned on (SST-2).

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        return self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)

    def predict(self, text: str) -> Dict[str, any]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            confidence, class_idx = torch.max(probs, dim=-1)

        label = self.model.config.id2label[class_idx.item()]
        return {"label": label, "probability": confidence.item()}

In [6]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "This tutorial is extremely helpful and well-structured!",
    "The documentation was confusing and led to many errors."
]
for text in samples:
    res = analyzer.predict(text)
    print(f"Text: {text}\nPrediction: {res}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This tutorial is extremely helpful and well-structured!
Prediction: {'label': 'POSITIVE', 'probability': 0.9998695850372314}

Text: The documentation was confusing and led to many errors.
Prediction: {'label': 'NEGATIVE', 'probability': 0.9996604919433594}



## Exercise 4 - BERT for Named Entity Recognition

### Deliverables
- **Subword Handling**: Tokens starting with `##` (like `##rin` in `Brin`) indicate that the original word was fragmented by the WordPiece algorithm. For clean NER, these tokens should normally inherit the label of the first part of the word or be ignored during metric calculations to avoid counting an entity multiple times.

In [13]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def recognize(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=2)
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        entities = []
        # Corrected: matching the variable name 'prediction' consistently and removed syntax noise
        for token, prediction in zip(tokens, predictions[0].tolist()):
            label = self.model.config.id2label[prediction]
            if label != 'O':
                entities.append({"token": token, "label": label})
        return entities

In [8]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin in California."
results = ner.recognize(sample_text)
for item in results:
    print(item)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'token': 'Google', 'label': 'B-ORG'}
{'token': 'Larry', 'label': 'B-PER'}
{'token': 'Page', 'label': 'I-PER'}
{'token': 'Sergey', 'label': 'B-PER'}
{'token': 'B', 'label': 'I-PER'}
{'token': '##rin', 'label': 'I-PER'}
{'token': 'California', 'label': 'B-LOC'}


## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder only (Bidirectional) | Decoder only (Unidirectional) |
| Primary purpose | Natural Language Understanding (NLU) | Natural Language Generation (NLG) |
| Typical use cases | Classification, NER, Question-Answering | Chatbots, Writing, Code Completion |
| Strengths | Full bidirectional context | Creativity and generation fluidity |
| Weaknesses | Not suited for long text generation | Less efficient for fine bidirectional analysis |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Encoding**: BERT transforms documents and queries into dense vectors (embeddings). Thanks to its bidirectional nature, it captures the deep semantic meaning of each sentence, allowing a question to be compared to thousands of documents not just by keywords, but by meaning.

2. **Vector DB**: These embeddings are stored in a vector database (like Pinecone or Milvus). When a query arrives, it is encoded by BERT and a 'nearest neighbor' search (Cosine Similarity) is performed to find the most relevant passages.

3. **Hand-off to GPT**: The passages extracted by BERT are injected into the 'prompt' sent to the generative model (GPT). GPT then uses this factual context to write an accurate response, thus limiting hallucinations.

4. **Application**: An internal technical support system for an aerospace company. BERT indexes complex technical manuals, and GPT answers engineers' questions based exclusively on the data extracted from the manual.